![Week 2 Day 3](../../assets/banners/w2d3.png)

# Week 2 Day 3 - SOLUTION - Filters, IDs and Real Re-indexing

Reference solution for the exercise at the end of `week2/day3.ipynb`. Have a go yourself first: struggling a little is where the learning happens! Then compare.

**The exercise:** write a filter for Unit 1 **or** Unit 3 on page 4+; prove stable IDs prevent duplicates (and random IDs don't); make a real content change to a copy of a PDF and check `ingest()` re-indexes it.

💡 **Fast & cheap run.** Two lines in `.env` decide what this notebook costs. `MODEL_MODE=free` runs on OpenRouter's free models for $0 (same `OPENROUTER_API_KEY`, no credits, but only about 50 requests a day), and `DATA_SIZE=small` uses a small slice of the data (2 PDFs · Q1–Q3 · 2 students) when you just want to see the APIs work. Restart the kernel after editing `.env`; the next cell prints what you're running.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 🔑 cheap and free both use one key, OPENROUTER_API_KEY (from openrouter.ai/keys); custom may need others
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"ℹ️ {key_name} not set. That's OK for most of this lab: the offline cells use fakes.")
        print("   The few cells that call a real model will tell you so. Setup: Part 5 of the setup guide.")

print(settings.describe_models())
print(datasets.describe())

In [ ]:
import shutil
import tempfile
import uuid
from pathlib import Path

import pymupdf
from langchain_core.embeddings import DeterministicFakeEmbedding

from gradewise.config import DATA_DIR
from gradewise.ingestion import chunk_documents, get_vectorstore, ingest, load_pdf

KB = DATA_DIR / "knowledge-base"
# ⚠️ Offline: fake embeddings (meaningless similarity) in a temp folder; real ones: gradewise.models.embeddings()
fake = DeterministicFakeEmbedding(size=64)
TMP = Path(tempfile.mkdtemp(prefix="gradewise_w2d3_solution_"))

chunks = chunk_documents([page for pdf in datasets.kb_files(KB) for page in load_pdf(pdf)])  # 💡 DATA_SIZE picks
store = get_vectorstore("cs201", embeddings=fake, persist_dir=TMP / "store")
ids = [f"{c.metadata['source']}#{c.metadata['chunk_index']}" for c in chunks]
store.add_documents(chunks, ids=ids)
print(store._collection.count(), "chunks stored")

## Part 1: Unit 1 or Unit 3, page 4 or later

In [ ]:
where = {"$and": [{"$or": [{"unit": 1}, {"unit": 3}]}, {"page": {"$gte": 4}}]}
# 💡 equivalently: {"$and": [{"unit": {"$in": [1, 3]}}, {"page": {"$gte": 4}}]}
found = store.get(where=where, include=["metadatas"])
pairs = sorted({(m["unit"], m["page"]) for m in found["metadatas"]})
print(f"{len(found['ids'])} chunks; (unit, page) pairs: {pairs}")
assert all(u in (1, 3) and p >= 4 for u, p in pairs)

## Part 2: Stable IDs versus random IDs

In [ ]:
before = store._collection.count()
store.add_documents(chunks, ids=ids)  # same IDs → overwrite
print(f"Same IDs again  : {before} → {store._collection.count()} chunks")

store.add_documents(chunks, ids=[str(uuid.uuid4()) for _ in chunks])  # new IDs → duplicates!
print(f"Random IDs again: {before} → {store._collection.count()} chunks  ← every chunk is now stored twice")

With random IDs, every search would now return each chunk twice, wasting half of every top-k. That's why `ingest()` builds IDs from the file name, its hash and the chunk number.

## Part 3: A real content change

In [ ]:
kb_copy = TMP / "kb"
shutil.copytree(KB, kb_copy)
db = TMP / "ingest"
first = ingest(kb_dir=kb_copy, collection="cs201", embeddings=fake, persist_dir=db, quiet=True)

# ✏️ Emma adds a sentence to page 1 of Unit 1, and saves a new version of the PDF
#    (💡 Unit 1 is in the knowledge base with both DATA_SIZE=full and small)
pdf = kb_copy / "unit1_operating_systems.pdf"
with pymupdf.open(pdf) as doc:
    doc[0].insert_text((72, 760), "Errata: the deadlock example in 1.6 now uses three processes.", fontsize=9)
    doc.save(TMP / "edited.pdf")
shutil.move(TMP / "edited.pdf", pdf)

second = ingest(kb_dir=kb_copy, collection="cs201", embeddings=fake, persist_dir=db, quiet=True)
print("updated:", second.updated, "· skipped:", second.skipped)
print(f"chunks: {first.chunks_total} before, {second.chunks_total} after (old chunks deleted, new ones added)")

store2 = get_vectorstore("cs201", embeddings=fake, persist_dir=db)
hits = store2.get(where={"source": "unit1_operating_systems.pdf"}, include=["documents"])["documents"]
print("New sentence is searchable:", any("Errata" in text for text in hits))

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">The delete-before-add logic you just exercised is in <code>gradewise/ingestion/pipeline.py</code>, inside <code>ingest()</code>.</span>
        </td>
    </tr>
</table>